# Diff-ICMH — bước 1: baseline checkpoint gốc trên Colab

Chọn GPU T4 và runtime **2026.07 (Python 3.12)**, rồi chạy **Runtime → Run all**. Lượt mặc định dùng BPP_WEIGHT=2, crop giữa 256×256 của `kodim01.png`, 10 bước DDPM để kiểm tra nạp checkpoint, nén, giải nén và giải mã. Sau khi thành công, đặt `SMOKE_TEST=False` để chạy 50 bước. Notebook tạo `baseline_report.json`, ảnh tái tạo, bitstream và ZIP. Đây chưa phải đánh giá Camera Trap.


## 1. Kiểm tra GPU, dung lượng và kết nối

In [ ]:
from pathlib import Path
import os
import platform
import shutil
import subprocess
import sys
from urllib.request import Request, urlopen

import torch
import torchvision

WORKING_DIR = Path('/content')
if not WORKING_DIR.is_dir():
    raise RuntimeError('Run this notebook on Google Colab.')
if sys.version_info[:2] != (3, 12):
    raise RuntimeError('Select Runtime > Change runtime type > Runtime version 2026.07 (Python 3.12), then rerun.')
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Runtime > Change runtime type, then rerun.')

GPU_NAME = torch.cuda.get_device_name(0)
GPU_GIB = torch.cuda.get_device_properties(0).total_memory / 2**30
free_gib = shutil.disk_usage(WORKING_DIR).free / 2**30
if GPU_GIB < 14:
    raise RuntimeError(f'{GPU_NAME} has {GPU_GIB:.1f} GiB VRAM; this full checkpoint needs a T4/L4 class GPU.')
if free_gib < 24:
    raise RuntimeError(f'Only {free_gib:.1f} GiB free; three checkpoints and temporary files need at least 24 GiB.')

for url in ('https://github.com', 'https://huggingface.co'):
    with urlopen(Request(url, headers={'User-Agent': 'Wild-Diff-ICMH-Colab-baseline/1.0'}), timeout=20) as response:
        if response.status >= 400:
            raise RuntimeError(f'HTTPS preflight failed: {url}, status={response.status}')

BASE_TORCH_VERSION = torch.__version__
BASE_TORCHVISION_VERSION = torchvision.__version__
BASE_CUDA_VERSION = torch.version.cuda
print('Python:', sys.version.split()[0], platform.platform())
print('Torch:', BASE_TORCH_VERSION, 'Torchvision:', BASE_TORCHVISION_VERSION)
print('CUDA ABI:', BASE_CUDA_VERSION, 'GPU:', GPU_NAME, f'{GPU_GIB:.1f} GiB')
print(f'Free disk: {free_gib:.1f} GiB')
subprocess.run(['nvidia-smi'], check=False)


## 2. Lấy đúng phiên bản mã nguồn đã kiểm tra

In [ ]:
REPO_URL = 'https://github.com/RuoyuFeng/Diff-ICMH.git'
REPO_REF = '01366f0afe8983eab423bb9a804beb2e02043100'
REPO_DIR = WORKING_DIR / 'Wild-Diff-ICMH'

def run_git(*args):
    return subprocess.run(['git', *args], cwd=REPO_DIR if REPO_DIR.exists() else WORKING_DIR, check=True, text=True)

if REPO_DIR.exists() and not (REPO_DIR / '.git').is_dir():
    raise RuntimeError(f'{REPO_DIR} exists but is not a Git repository; remove or rename it before retrying.')
if not REPO_DIR.exists():
    subprocess.run(
        ['git', 'clone', '--filter=blob:none', '--no-checkout', REPO_URL, str(REPO_DIR)],
        cwd=WORKING_DIR,
        check=True,
    )

run_git('remote', 'set-url', 'origin', REPO_URL)
run_git('fetch', '--depth', '1', 'origin', REPO_REF)
run_git('checkout', '--detach', '--force', REPO_REF)
checked_out_ref = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True
).strip()
if checked_out_ref != REPO_REF:
    raise RuntimeError(f'Expected {REPO_REF}, but checked out {checked_out_ref}.')

print('Repository:', REPO_DIR)
print('Pinned revision:', checked_out_ref)

## 3. Cài thư viện và kiểm tra import

In [ ]:
def pip_install(*packages, extra_args=()):
    command = [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '-q', *extra_args, *packages]
    print('Installing:', ' '.join(packages))
    subprocess.run(command, check=True)

# Runtime-safe replacements for stale repository pins.
pip_install(
    'numpy>=1.26,<2.0',
    'lightning>=2.6,<3',
    'pyiqa==0.1.15.post2',
    'fairscale==0.4.13',
    'einops>=0.8,<1',
    'kornia>=0.7,<1',
    'omegaconf==2.3.0',
    'open_clip_torch>=2.22,<3',
    'openai_clip>=1.0.1,<2',
    'opencv-python-headless>=4.8,<5',
    'Pillow>=10.4,<13',
    'pytorch-msssim>=1.0,<2',
    'scipy>=1.11,<2',
    'matplotlib>=3.8,<4',
    'tomli>=2,<3',
    'thop>=0.1.1.post2209072238',
    'timm>=0.9.7,<1.0',
    'tqdm>=4.66,<5',
    'transformers>=4.41,<5',
    'ultralytics_thop>=2,<3',
    'huggingface_hub>=0.23,<2',
)
pip_install('torch-geometric>=2.6,<3')
pip_install('compressai==1.2.8', extra_args=('--no-deps',))
pip_install('-e', str(REPO_DIR / 'src' / 'recognize-anything'), extra_args=('--no-deps',))

if not BASE_CUDA_VERSION:
    raise RuntimeError(f'Torch {BASE_TORCH_VERSION} reports no CUDA ABI even though a GPU was detected.')
cuda_parts = BASE_CUDA_VERSION.split('.')
if len(cuda_parts) < 2 or not all(part.isdigit() for part in cuda_parts[:2]):
    raise RuntimeError(f'Cannot derive a PyTorch wheel index from CUDA ABI {BASE_CUDA_VERSION!r}.')
torch_wheel_tag = f'cu{cuda_parts[0]}{cuda_parts[1]}'
torch_wheel_index = f'https://download.pytorch.org/whl/{torch_wheel_tag}'
try:
    pip_install('xformers', extra_args=('--index-url', torch_wheel_index, '--no-deps'))
    subprocess.run(
        [sys.executable, '-c', 'import xformers, xformers.ops; print("xFormers:", xformers.__version__)'],
        check=True,
    )
except subprocess.CalledProcessError as exc:
    raise RuntimeError(
        'No working xFormers wheel was established for the detected Colab ABI. '
        f'Torch={BASE_TORCH_VERSION}, Torchvision={BASE_TORCHVISION_VERSION}, '
        f'CUDA={BASE_CUDA_VERSION}, index={torch_wheel_index}. '
        'Start a current Colab GPU runtime whose Torch/CUDA pair has a published PyTorch xFormers wheel, then retry.'
    ) from exc

In [ ]:
import json
import textwrap

COMPAT_DIR = WORKING_DIR / 'difficmh_compat'
PL_DIR = COMPAT_DIR / 'pytorch_lightning'
UTILITIES_DIR = PL_DIR / 'utilities'
CALLBACKS_DIR = PL_DIR / 'callbacks'
for directory in (COMPAT_DIR, PL_DIR, UTILITIES_DIR, CALLBACKS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

def write_compat(relative_path, content):
    target = COMPAT_DIR / relative_path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(textwrap.dedent(content).lstrip(), encoding='utf-8')
    return target

write_compat('pytorch_lightning/__init__.py', '''
    from lightning.pytorch import LightningDataModule, LightningModule, Trainer, seed_everything
    from lightning.pytorch import __version__

    __all__ = [
        'LightningDataModule', 'LightningModule', 'Trainer', 'seed_everything', '__version__'
    ]
''')
write_compat('pytorch_lightning/utilities/__init__.py', '''
    from .distributed import rank_zero_only
    from .types import EPOCH_OUTPUT, STEP_OUTPUT

    __all__ = ['rank_zero_only', 'EPOCH_OUTPUT', 'STEP_OUTPUT']
''')
write_compat('pytorch_lightning/utilities/distributed.py', '''
    from lightning.pytorch.utilities.rank_zero import rank_zero_only

    __all__ = ['rank_zero_only']
''')
write_compat('pytorch_lightning/utilities/types.py', '''
    from typing import Any, Dict, List, Mapping, Optional, Sequence, Union

    STEP_OUTPUT = Optional[Union[Dict[str, Any], Any]]
    EPOCH_OUTPUT = List[STEP_OUTPUT]

    __all__ = ['STEP_OUTPUT', 'EPOCH_OUTPUT']
''')
write_compat('pytorch_lightning/callbacks/__init__.py', '''
    from lightning.pytorch.callbacks import *
''')
write_compat('lpips.py', '''
    import torch
    import pyiqa

    class LPIPS(torch.nn.Module):
        def __init__(self, net='alex'):
            super().__init__()
            self.model = pyiqa.create_metric('lpips', net=net, device='cpu')

        def forward(self, img1, img2, normalize=False):
            # pyiqa's public metric API expects [0, 1]. Legacy lpips expects
            # [-1, 1] unless normalize=True, so convert only that legacy case.
            if not normalize:
                img1 = (img1 + 1.0) / 2.0
                img2 = (img2 + 1.0) / 2.0
            return self.model(img1, img2)
''')

compat_env = os.environ.copy()
pythonpath_parts = [str(COMPAT_DIR), str(REPO_DIR)]
if compat_env.get('PYTHONPATH'):
    pythonpath_parts.append(compat_env['PYTHONPATH'])
compat_env['PYTHONPATH'] = os.pathsep.join(pythonpath_parts)
compat_env['TOKENIZERS_PARALLELISM'] = 'false'

version_probe = textwrap.dedent(f'''
    import json, torch, torchvision
    import lightning, compressai, pyiqa, xformers
    assert torch.__version__ == {BASE_TORCH_VERSION!r}, (torch.__version__, {BASE_TORCH_VERSION!r})
    assert torchvision.__version__ == {BASE_TORCHVISION_VERSION!r}, (torchvision.__version__, {BASE_TORCHVISION_VERSION!r})
    print(json.dumps({{
        'torch': torch.__version__,
        'torchvision': torchvision.__version__,
        'cuda': torch.version.cuda,
        'lightning': lightning.__version__,
        'compressai': compressai.__version__,
        'pyiqa': pyiqa.__version__,
        'xformers': xformers.__version__,
    }}, indent=2))
''')
subprocess.run([sys.executable, '-c', version_probe], cwd=REPO_DIR, env=compat_env, check=True)
subprocess.run(
    [sys.executable, '-c', 'import inference_partition; print("inference_partition import smoke check: OK")'],
    cwd=REPO_DIR,
    env=compat_env,
    check=True,
)

## 4. Giảm bộ nhớ T4 và giữ SD 2.1 đóng băng

In [ ]:
# Cell 5 — Compatibility and T4 memory patches
from pathlib import Path

REPO = REPO_DIR

def replace_once(relative_path, old, new, label):
    path = REPO / relative_path
    if not path.is_file():
        raise FileNotFoundError(f"Repo source missing: {path}. Rerun from Cell 2.")
    source = path.read_text(encoding="utf-8")
    if new in source:
        print(f"OK (already patched): {label}")
        return
    if old not in source:
        raise RuntimeError(f"Source changed; cannot apply {label}: {path}")
    path.write_text(source.replace(old, new, 1), encoding="utf-8")
    print(f"OK (patched): {label}")

replace_once(
    "model/diffeic.py",
    "from pytorch_lightning.utilities.types import EPOCH_OUTPUT",
    """try:
    from pytorch_lightning.utilities.types import EPOCH_OUTPUT
except ImportError:
    from typing import Any
    EPOCH_OUTPUT = Any""",
    "Lightning EPOCH_OUTPUT",
)

for relative_path in ["ldm/models/diffusion/ddpm.py", "model/callbacks.py"]:
    replace_once(
        relative_path,
        "from pytorch_lightning.utilities.distributed import rank_zero_only",
        "from pytorch_lightning.utilities.rank_zero import rank_zero_only",
        f"Lightning rank_zero_only in {relative_path}",
    )

replace_once(
    "ldm/modules/encoders/modules.py",
    "model, _, _ = open_clip.create_model_and_transforms(arch, device=torch.device('cpu'), pretrained=version)",
    """# SD 2.1 checkpoint supplies these text encoder weights.
        model, _, _ = open_clip.create_model_and_transforms(
            arch, device=torch.device('cpu'), pretrained=None
        )""",
    "skip redundant OpenCLIP download",
)

old_loader = """    ckpt_sd = torch.load(args.ckpt_sd, map_location="cpu")['state_dict']
    ckpt_lc = torch.load(args.ckpt_lc, map_location="cpu")['state_dict']
    ckpt_sd.update(ckpt_lc)
    msg = load_state_dict(model, ckpt_sd, strict=False)
    print(f"Messgae of load state dict: {msg}")"""

new_loader = """    # Load checkpoints sequentially so both full state dicts are never resident together.
    import gc

    def load_checkpoint(path, label):
        print(f"Loading {label}: {path}")
        try:
            checkpoint = torch.load(path, map_location="cpu", mmap=True, weights_only=False)
        except TypeError:  # older PyTorch without mmap/weights_only
            checkpoint = torch.load(path, map_location="cpu")
        state_dict = checkpoint["state_dict"]
        message = load_state_dict(model, state_dict, strict=False)
        print(f"{label}: missing={len(message.missing_keys)}, unexpected={len(message.unexpected_keys)}")
        del state_dict, checkpoint
        gc.collect()

    load_checkpoint(args.ckpt_sd, "Stable Diffusion 2.1")
    load_checkpoint(args.ckpt_lc, "Diff-ICMH BPP=2")"""

replace_once("inference_partition.py", old_loader, new_loader, "sequential checkpoint loading")

replace_once(
    "inference_partition.py",
    "model.preprocess_tag_model(control, return_ids=True)",
    "model.preprocess_tag_model(control.cpu(), return_ids=True)",
    "run RAM+ on CPU",
)

replace_once(
    "inference_partition.py",
    "    model.freeze()\n    model.to(args.device)",
    """    model.freeze()
    assert all(not p.requires_grad for p in model.model.diffusion_model.parameters()), "SD UNet is not frozen"
    # Temporarily detach RAM+ so model.to(cuda) never copies its 3 GiB to T4.
    tagger = model.preprocess_tag_model
    if args.device == "cuda" and tagger.enabled:
        del model._modules["preprocess_tag_model"]
        model.to(args.device)
        model.preprocess_tag_model = tagger.to("cpu")
    else:
        model.to(args.device)""",
    "keep RAM+ off T4 VRAM",
)

print("Compatibility patches ready.")

## 5. Tải đúng ba checkpoint cho một mức nén

In [ ]:
from huggingface_hub import hf_hub_download

BPP_WEIGHT = 2
CONTROL_MODULE_SCALE = 1.0
CFG_SCALE = 5.0
SMOKE_TEST = True
STEPS = 10 if SMOKE_TEST else 50
SEED = 231

if BPP_WEIGHT not in {2, 4, 8, 16, 32}:
    raise ValueError('BPP_WEIGHT must be one of 2, 4, 8, 16, or 32.')
FOLDER_NAME = f'CNscale1.0_1_1_{BPP_WEIGHT}_2_WTagGCM_bs16x1_lr0.00005_cfg7.0'

CHECKPOINTS_DIR = REPO_DIR / 'checkpoints'
SD_DIR = CHECKPOINTS_DIR / 'sd2p1'
RAM_DIR = CHECKPOINTS_DIR / 'ram'
SD_DIR.mkdir(parents=True, exist_ok=True)
RAM_DIR.mkdir(parents=True, exist_ok=True)

hf_hub_download(
    repo_id='Manojb/stable-diffusion-2-1-base',
    filename='v2-1_512-ema-pruned.ckpt',
    local_dir=SD_DIR,
)
hf_hub_download(
    repo_id='xinyu1205/recognize-anything-plus-model',
    filename='ram_plus_swin_large_14m.pth',
    local_dir=RAM_DIR,
)
hf_hub_download(
    repo_id='RuoyuFeng/Diff-ICMH',
    filename=f'difficmh_models/{FOLDER_NAME}/model.ckpt',
    local_dir=CHECKPOINTS_DIR,
)

CKPT_SD = SD_DIR / 'v2-1_512-ema-pruned.ckpt'
CKPT_RAM = RAM_DIR / 'ram_plus_swin_large_14m.pth'
CKPT_LC = CHECKPOINTS_DIR / 'difficmh_models' / FOLDER_NAME / 'model.ckpt'
minimum_sizes = {CKPT_SD: 1_000_000_000, CKPT_RAM: 100_000_000, CKPT_LC: 10_000_000}
for checkpoint, minimum_size in minimum_sizes.items():
    if not checkpoint.is_file() or checkpoint.stat().st_size < minimum_size:
        actual = checkpoint.stat().st_size if checkpoint.exists() else 0
        raise RuntimeError(
            f'Checkpoint is missing or unexpectedly small: {checkpoint} ({actual:,} bytes; expected at least {minimum_size:,}).'
        )
    print(f'Checkpoint OK: {checkpoint.relative_to(REPO_DIR)} ({checkpoint.stat().st_size / 2**30:.2f} GiB)')

In [ ]:
from omegaconf import OmegaConf

BASE_CONFIG = REPO_DIR / 'configs' / 'model' / 'diffeic.yaml'
COLAB_CONFIG = REPO_DIR / 'configs' / 'model' / f'diffeic_colab_bpp{BPP_WEIGHT}.yaml'
cfg = OmegaConf.load(BASE_CONFIG)
# The CLI loads SD and Diff-ICMH checkpoints sequentially. Avoid a third SD load in __init__.
cfg.params.sync_path = None
cfg.params.synch_control = False
cfg.params.control_stage_config.params.control_model_ratio = CONTROL_MODULE_SCALE
cfg.params.preprocess_semantic_config.params.enabled = False
cfg.params.preprocess_tag_config.params.enabled = True
cfg.params.preprocess_tag_config.params.pretrained = str(CKPT_RAM)
cfg.params.c_cfg_scale = CFG_SCALE
cfg.params.calculate_metrics = {}
OmegaConf.save(cfg, COLAB_CONFIG)
print('Inference config:', COLAB_CONFIG)


## 6. Tạo một ảnh thử cố định từ Kodak

In [ ]:
from PIL import Image

# This is a checkpoint/load smoke test, not a camera-trap benchmark.
SOURCE_IMAGE = REPO_DIR / 'data' / 'kodak_subset' / 'kodim01.png'
if not SOURCE_IMAGE.is_file():
    raise FileNotFoundError(SOURCE_IMAGE)
INPUT_DIR = WORKING_DIR / 'difficmh_baseline_input'
OUTPUT_DIR = WORKING_DIR / f'difficmh_baseline_bpp{BPP_WEIGHT}_{STEPS}steps'
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for old in INPUT_DIR.iterdir():
    if old.is_file():
        old.unlink()

with Image.open(SOURCE_IMAGE) as source_image:
    source_rgb = source_image.convert('RGB')
    width, height = source_rgb.size
    crop_size = 256
    left, top = (width - crop_size) // 2, (height - crop_size) // 2
    if left < 0 or top < 0:
        raise RuntimeError(f'{SOURCE_IMAGE} is smaller than 256x256')
    smoke_crop = source_rgb.crop((left, top, left + crop_size, top + crop_size))

STAGED_IMAGE = INPUT_DIR / 'kodim01_center256.png'
smoke_crop.save(STAGED_IMAGE)
print('Source:', SOURCE_IMAGE, (width, height))
print('Staged 256x256 crop:', STAGED_IMAGE)


## 7. Chạy baseline gốc

In [ ]:
# The first pass checks full model loading, compression, decompression, and decoding.
# For a quality baseline after this succeeds, set SMOKE_TEST=False above (50 steps).
env = compat_env.copy()
env['CUDA_VISIBLE_DEVICES'] = '0'
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
env['OMP_NUM_THREADS'] = '2'
command = [
    sys.executable, '-u', 'inference_partition.py',
    '--ckpt_sd', str(CKPT_SD), '--ckpt_lc', str(CKPT_LC),
    '--config', str(COLAB_CONFIG), '--input', str(INPUT_DIR),
    '--output', str(OUTPUT_DIR), '--sampler', 'ddpm',
    '--steps', str(STEPS), '--seed', str(SEED), '--device', 'cuda',
]
print('Running:', command)
subprocess.run(command, check=True, cwd=REPO_DIR, env=env)


## 8. Kiểm tra ảnh, bitstream và chỉ số

In [ ]:
import json
import textwrap
from IPython.display import display
from PIL import Image

RECONSTRUCTION = OUTPUT_DIR / f'{STAGED_IMAGE.stem}.png'
BITSTREAM = OUTPUT_DIR / 'data' / STAGED_IMAGE.stem
CLI_METRICS = OUTPUT_DIR / 'bpp.txt'
for item in (RECONSTRUCTION, BITSTREAM, CLI_METRICS):
    if not item.is_file() or item.stat().st_size == 0:
        raise RuntimeError(f'Missing or empty baseline output: {item}')
with Image.open(STAGED_IMAGE) as original, Image.open(RECONSTRUCTION) as decoded:
    if decoded.size != original.size:
        raise RuntimeError(f'Reconstruction size {decoded.size} != input {original.size}')
    width, height = original.size
    display(original, decoded)

metric_program = textwrap.dedent('''
    import json, sys, pyiqa, torch
    from PIL import Image
    from torchvision.transforms.functional import pil_to_tensor
    image_paths = sys.argv[1:3]
    tensors = [pil_to_tensor(Image.open(p).convert('RGB')).unsqueeze(0).float().cuda()/255 for p in image_paths]
    results = {name: float(pyiqa.create_metric(name, device='cuda')(*tensors).item())
               for name in ('psnr', 'ssim', 'lpips')}
    print('METRICS_JSON=' + json.dumps(results, sort_keys=True))
''')
metric_run = subprocess.run(
    [sys.executable, '-c', metric_program, str(STAGED_IMAGE), str(RECONSTRUCTION)],
    cwd=REPO_DIR, env=compat_env, check=True, text=True, capture_output=True,
)
metric_line = next((line for line in metric_run.stdout.splitlines() if line.startswith('METRICS_JSON=')), None)
if metric_line is None:
    raise RuntimeError(f'Metric subprocess returned no JSON: {metric_run.stdout} {metric_run.stderr}')
metrics = json.loads(metric_line.split('=', 1)[1])
actual_bpp = 8 * BITSTREAM.stat().st_size / (width * height)
baseline = {
    'status': 'gpu_verified_baseline' if not SMOKE_TEST else 'gpu_verified_smoke',
    'repository_commit': checked_out_ref,
    'checkpoint_repo': 'RuoyuFeng/Diff-ICMH',
    'checkpoint_relative_path': f'difficmh_models/{FOLDER_NAME}/model.ckpt',
    'sd_checkpoint': CKPT_SD.name,
    'bpp_weight': BPP_WEIGHT,
    'control_model_ratio': CONTROL_MODULE_SCALE,
    'tag_guidance_enabled': True,
    'sd_unet_frozen': True,
    'source_image': str(SOURCE_IMAGE.relative_to(REPO_DIR)),
    'input_transform': 'center crop 256x256',
    'input_pixels': [width, height],
    'sampler': 'ddpm', 'sampler_steps': STEPS, 'seed': SEED,
    'bitstream_bytes': BITSTREAM.stat().st_size,
    'bpp_actual': actual_bpp,
    **metrics,
}
REPORT_PATH = OUTPUT_DIR / 'baseline_report.json'
REPORT_PATH.write_text(json.dumps(baseline, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(json.dumps(baseline, indent=2, sort_keys=True))
print('CLI report:', CLI_METRICS.read_text(encoding='utf-8'))
print('Saved:', REPORT_PATH)


In [ ]:
from IPython.display import FileLink, display

archive_base = WORKING_DIR / f'difficmh_baseline_bpp{BPP_WEIGHT}_{STEPS}steps'
archive_path = Path(shutil.make_archive(str(archive_base), 'zip', root_dir=OUTPUT_DIR))
print('Downloadable baseline package:', archive_path)
display(FileLink(str(archive_path)))


**Trạng thái:** Notebook này được kiểm tra tĩnh tại workspace. Chỉ coi baseline hoàn tất sau khi Colab chạy đến `baseline_report.json` và có ảnh cùng bitstream không rỗng.